# Week 2 — Data Parallelism: From `DataParallel` to a Hand-Built DDP

This notebook re-derives the **synchronous SGD** estimator that underlies all data-parallel training, proves the bandwidth-optimality of **Ring-AllReduce**, and culminates in a from-scratch `DistributedDataParallel` implementation built on `torch.distributed` point-to-point primitives.

## Learning objectives

1. Show that data-parallel SGD with global batch $B = \sum_{i} B_i$ is an unbiased estimator of the gradient computed on the union of the local batches.
2. Derive the communication cost of Ring-AllReduce and prove it is bandwidth-optimal.
3. Explain why `nn.DataParallel` (DP) cannot scale and `DistributedDataParallel` (DDP) can.
4. Implement gradient bucketing and overlap with backward.


## 1. Why Data Parallelism Works — A One-Line Proof

Let the loss on a global batch $\mathcal{B} = \bigsqcup_{i=1}^{N} \mathcal{B}_i$ be

$$
L(\theta) \;=\; \frac{1}{|\mathcal{B}|} \sum_{x \in \mathcal{B}} \ell(\theta; x)
\;=\; \frac{1}{N} \sum_{i=1}^{N} \underbrace{\frac{1}{|\mathcal{B}_i|} \sum_{x \in \mathcal{B}_i} \ell(\theta; x)}_{L_i(\theta)} \quad \text{(when } |\mathcal{B}_i|=B/N\text{)}.
$$

By linearity of the gradient,

$$
\nabla_\theta L(\theta) \;=\; \frac{1}{N} \sum_{i=1}^{N} \nabla_\theta L_i(\theta).
$$

So the global gradient is the **mean of the local gradients** — exactly what an AllReduce-Mean computes. This is the entire mathematical content of data-parallel training; the rest is communication engineering.


## 2. Ring-AllReduce: Bandwidth-Optimality

Consider $N$ workers each holding a tensor of size $S$ bytes. We want every worker to end with the sum of all $N$ tensors.

### The two-phase algorithm

**Phase 1 — Reduce-Scatter.** Split each tensor into $N$ chunks of size $S/N$. Over $N-1$ rounds, worker $r$ sends chunk $((r - t) \bmod N)$ to its right neighbour and receives chunk $((r - t - 1) \bmod N)$ from its left neighbour, summing it into its local copy. After $N-1$ rounds, worker $r$ holds the **fully reduced** value of chunk $((r + 1) \bmod N)$.

**Phase 2 — All-Gather.** Over another $N-1$ rounds, each worker forwards its fully-reduced chunk around the ring until every worker holds every chunk.

### Per-worker byte count

Each phase moves $(N-1) \cdot (S/N)$ bytes per worker, so the total is

$$
C(N, S) \;=\; 2 (N-1) \cdot \frac{S}{N}.
$$

Crucially, as $N \to \infty$ this tends to $2S$ — **independent of $N$**. No collective can do better than $2S - 2S/N$ per worker, so Ring-AllReduce is asymptotically optimal.


In [ ]:
# %% From-scratch Ring-AllReduce on a single node with torch.distributed
# Run with:  torchrun --standalone --nproc_per_node=4 02_ddp_from_scratch.py
import os
import torch
import torch.distributed as dist

def init_distributed() -> tuple[int, int]:
    dist.init_process_group(backend='nccl')
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    torch.cuda.set_device(rank)
    return rank, world_size

def ring_allreduce(tensor: torch.Tensor) -> torch.Tensor:
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    left  = (rank - 1) % world_size
    right = (rank + 1) % world_size

    # Split tensor into world_size chunks.
    chunks = list(tensor.chunk(world_size))

    # ---- Phase 1: reduce-scatter ----
    for step in range(world_size - 1):
        send_idx = (rank - step) % world_size
        recv_idx = (rank - step - 1) % world_size

        recv_buf = torch.empty_like(chunks[recv_idx])
        send_op  = dist.P2POp(dist.isend, chunks[send_idx], right)
        recv_op  = dist.P2POp(dist.irecv, recv_buf, left)
        for req in dist.batch_isend_irecv([send_op, recv_op]):
            req.wait()
        chunks[recv_idx] += recv_buf

    # ---- Phase 2: all-gather ----
    for step in range(world_size - 1):
        send_idx = (rank - step + 1) % world_size
        recv_idx = (rank - step) % world_size
        recv_buf = torch.empty_like(chunks[recv_idx])
        send_op  = dist.P2POp(dist.isend, chunks[send_idx], right)
        recv_op  = dist.P2POp(dist.irecv, recv_buf, left)
        for req in dist.batch_isend_irecv([send_op, recv_op]):
            req.wait()
        chunks[recv_idx] = recv_buf

    return torch.cat(chunks)


## 3. Why `DataParallel` Cannot Scale

`torch.nn.DataParallel` operates from a **single Python process**:

1. The master rank scatters the input batch across $N$ GPUs.
2. Each GPU runs its forward pass.
3. The master rank gathers all outputs, computes the loss, and scatters gradients back.

This design has two fatal flaws:

* **Python GIL.** All $N$ replicas dispatch CUDA kernels from the same process, so the GIL serialises kernel launches.
* **Master-rank bottleneck.** The scatter/gather pattern is $O(N)$ on a single PCIe/NVLink path, not the $O(2(N-1)/N)$ of Ring-AllReduce.

DDP eliminates both: each rank is a **separate process**, and gradients are reduced via Ring-AllReduce.


## 4. Gradient Bucketing and Overlap

A naive DDP would call AllReduce once per parameter — but small tensors are bandwidth-inefficient. PyTorch's DDP groups gradients into ~25 MiB buckets and **fires the AllReduce as soon as the last gradient in a bucket is produced**, so communication overlaps with the rest of the backward pass.

The expected step time becomes

$$
t_{\text{step}} \;\approx\; \max\bigl(t_{\text{compute}},\; t_{\text{comm}}\bigr) \quad \text{instead of} \quad t_{\text{compute}} + t_{\text{comm}}.
$$


In [ ]:
# %% Custom DDP wrapper sketch
import torch.nn as nn
from hpcllmforge.parallelism.data.ddp_engine import DDPEngine

model = nn.Sequential(nn.Linear(1024, 4096), nn.GELU(), nn.Linear(4096, 1024)).cuda()
ddp = DDPEngine(model, bucket_size_mb=25.0)

optim = torch.optim.AdamW(ddp.parameters(), lr=1e-4)

for step in range(100):
    x = torch.randn(32, 1024, device='cuda')
    y = ddp(x).sum()
    y.backward()
    ddp.finalize_backward()
    optim.step()
    optim.zero_grad()


## 5. Exercises

1. Time the from-scratch Ring-AllReduce against `dist.all_reduce` (NCCL) for tensor sizes $\{1, 4, 16, 64, 256\}$ MiB. Plot bandwidth utilisation.
2. Profile a small Transformer with and without overlap (use `find_unused_parameters=True` to break overlap). Report the wallclock difference.
3. Theoretically derive the optimal bucket size given measured `t_compute_per_param` and per-byte `t_allreduce`.
